# 05 — Silver V2 incremental nacional

Reconstrução consolidada da conversa. Valide os esquemas dos arquivos DBC/Parquet, teste sem gravação e execute apenas no Lakehouse de origem. Não execute em paralelo.

In [ ]:
import json,hashlib,pandas as pd
from pathlib import Path
from uuid import uuid4
from pyspark.sql import functions as F
from pyspark.sql import types as T
EXECUTION_ID=str(uuid4())
ROOT=Path('/lakehouse/default/Files/media')
assert ROOT.exists(), 'Anexe lh_media_analytics como Lakehouse padrao'
# Execute primeiro como simulacao; confira Lakehouse antes de habilitar
APLICAR_GRAVACAO=False
CONFIRMO_LAKEHOUSE_ORIGINAL=False
MAX_ARQUIVOS_EXECUCAO=10 # None para todos
ARQUIVOS_POR_LOTE=3
FONTES_EXECUCAO=['SIM','SIH','CNES_ST','CNES_LT']
ORDEM=['SIM','SIH','CNES_ST','CNES_LT']
UFS={'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'}
TABELAS={'SIH':'dbo.silver_v2_sih_internacoes','SIM':'dbo.silver_v2_sim_obitos','CNES_ST':'dbo.silver_v2_cnes_estabelecimentos','CNES_LT':'dbo.silver_v2_cnes_leitos'}
GRUPOS={'SIH':'RD','SIM':'DO','CNES_ST':'ST','CNES_LT':'LT'}
PASTAS={'SIH':'sih','SIM':'sim','CNES_ST':'cnes','CNES_LT':'cnes'}
registros=[]
for fonte in ORDEM:
 for cp in (ROOT/'controle'/PASTAS[fonte]).glob('*.json'):
  try:
   j=json.loads(cp.read_text(encoding='utf-8'))
   uf=str(j.get('uf','')).upper();grupo=str(j.get('grupo','')).upper()
   if j.get('status')!='CONCLUIDO' or grupo!=GRUPOS[fonte] or uf not in UFS:continue
   ano=int(j['ano']);mes=int(j.get('mes') or 0)
   if ano not in (2023,2024,2025,2026): continue
   dest=Path(str(j.get('destino','')));chave=str(j.get('chave',''))
   if not dest.is_file() or not chave: continue
   aid=hashlib.sha256(chave.encode()).hexdigest()[:20]
   registros.append(dict(fonte=fonte,uf=uf,ano=ano,mes=mes,grupo=grupo,arquivo_id=aid,arquivo_bronze=str(dest),linhas_bronze=int(j.get('linhas') or 0)))
  except Exception as ex: print('Controle ignorado',cp.name,ex)
df_manifesto=pd.DataFrame(registros).drop_duplicates(['fonte','arquivo_id'])
assert not df_manifesto.empty,'Nenhum controle Bronze valido'
display(df_manifesto.groupby(['fonte','ano'],as_index=False).agg(arquivos=('arquivo_id','count'),linhas=('linhas_bronze','sum')))


In [ ]:
def origem(df,*nomes):
 m={c.upper():c for c in df.columns}
 for n in nomes:
  if n.upper() in m:return F.col('`'+m[n.upper()]+'`')
 return F.lit(None)
def texto(df,*ns):
 c=F.trim(origem(df,*ns).cast('string'))
 return F.when(c.isNull()|(c==''),F.lit(None).cast('string')).otherwise(c)
def inteiro(df,*ns):return texto(df,*ns).try_cast('int')
def decimal(df,*ns):return F.regexp_replace(texto(df,*ns),',','.').try_cast('double')
def municipio(df,*ns):
 c=F.regexp_replace(texto(df,*ns),r'[^0-9]','')
 return F.when(F.length(c).isin(6,7),F.substring(c,1,6)).otherwise(F.lit(None).cast('string'))
def cid(df,*ns):return F.regexp_replace(F.upper(texto(df,*ns)),r'[^A-Z0-9]','')
def data(df,*ns):
 c=texto(df,*ns)
 return F.coalesce(*[F.try_to_timestamp(c,F.lit(fmt)).cast('date') for fmt in ('yyyyMMdd','ddMMyyyy','yyyy-MM-dd','yyyy-MM-dd HH:mm:ss')])
def meta(df,it):
 for col,val in [('arquivo_id',str(it['arquivo_id'])),('uf_arquivo',str(it['uf'])),('ano_arquivo',int(it['ano'])),('mes_arquivo',int(it['mes'])),('fonte','CNES' if it['fonte'].startswith('CNES') else it['fonte']),('grupo',it['grupo']),('execution_id',EXECUTION_ID)]:df=df.withColumn(col,F.lit(val))
 return df.withColumn('processado_em',F.current_timestamp())
def transformar_sih(df,it):
 d=df.select(texto(df,'N_AIH','numero_aih').alias('numero_aih'),texto(df,'CNES','cnes_estabelecimento').alias('cnes_estabelecimento'),municipio(df,'MUNIC_RES','municipio_residencia').alias('municipio_residencia'),municipio(df,'MUNIC_MOV','municipio_internacao').alias('municipio_internacao'),cid(df,'DIAG_PRINC','cid_principal').alias('cid_principal'),cid(df,'DIAG_SECUN','cid_secundario').alias('cid_secundario'),texto(df,'SEXO','sexo_codigo').alias('sexo_codigo'),inteiro(df,'IDADE','idade_valor').alias('idade_valor'),texto(df,'COD_IDADE','idade_unidade_codigo').alias('idade_unidade_codigo'),data(df,'DT_INTER','data_internacao').alias('data_internacao'),data(df,'DT_SAIDA','data_saida').alias('data_saida'),inteiro(df,'DIAS_PERM','dias_permanencia').alias('dias_permanencia'),decimal(df,'VAL_TOT','valor_total').alias('valor_total'),decimal(df,'VAL_UTI','valor_uti').alias('valor_uti'),inteiro(df,'MORTE','indicador_obito').alias('indicador_obito'),inteiro(df,'ANO_CMPT','ano_competencia').alias('ano_competencia'),inteiro(df,'MES_CMPT','mes_competencia').alias('mes_competencia'))
 d=d.withColumn('ano_competencia',F.coalesce('ano_competencia',F.lit(int(it['ano'])))).withColumn('mes_competencia',F.coalesce('mes_competencia',F.lit(int(it['mes']))))
 d=d.withColumn('doenca_cardiovascular',F.when(F.col('cid_principal').rlike(r'^I[0-9]{2}'),1).otherwise(0)).withColumn('capitulo_cid',F.substring('cid_principal',1,1))
 return meta(d,it)
def transformar_sim(df,it):
 d=df.select(texto(df,'NUMERODO','numero_do').alias('numero_do'),data(df,'DTOBITO','data_obito').alias('data_obito'),data(df,'DTNASC','data_nascimento').alias('data_nascimento'),municipio(df,'CODMUNRES','municipio_residencia').alias('municipio_residencia'),municipio(df,'CODMUNOCOR','municipio_ocorrencia').alias('municipio_ocorrencia'),cid(df,'CAUSABAS','cid_causa_basica').alias('cid_causa_basica'),texto(df,'CAUSABAS','cid_causa_basica_original').alias('cid_causa_basica_original'),texto(df,'SEXO','sexo_codigo').alias('sexo_codigo'),texto(df,'IDADE','idade_codigo').alias('idade_codigo'),texto(df,'RACACOR','raca_cor_codigo').alias('raca_cor_codigo'),texto(df,'ESC','escolaridade_codigo').alias('escolaridade_codigo'),texto(df,'CIRCOBITO','circunstancia_obito').alias('circunstancia_obito'),texto(df,'TIPOBITO','tipo_obito').alias('tipo_obito'))
 d=d.withColumn('ano_obito',F.year('data_obito')).withColumn('mes_obito',F.month('data_obito')).withColumn('codigo_uf_residencia',F.substring('municipio_residencia',1,2)).withColumn('doenca_cardiovascular',F.when(F.col('cid_causa_basica').rlike(r'^I[0-9]{2}'),1).otherwise(0))
 return meta(d,it)
def transformar_st(df,it):
 d=df.select(texto(df,'CNES').alias('cnes_estabelecimento'),municipio(df,'CODUFMUN').alias('municipio_codigo_6'),texto(df,'FANTASIA').alias('nome_fantasia'),texto(df,'TP_UNID').alias('tipo_unidade'),texto(df,'TP_GESTAO').alias('tipo_gestao'),texto(df,'PF_PJ').alias('tipo_pessoa'),texto(df,'NATUREZA').alias('natureza_juridica'))
 return meta(d,it)
def transformar_lt(df,it):
 d=df.select(texto(df,'CNES').alias('cnes_estabelecimento'),municipio(df,'CODUFMUN').alias('municipio_codigo_6'),texto(df,'CODLEITO','COD_LEITO').alias('codigo_leito'),texto(df,'TP_LEITO').alias('tipo_leito'),inteiro(df,'QT_EXIST').alias('quantidade_existente'),inteiro(df,'QT_SUS').alias('quantidade_sus'))
 return meta(d,it)
TRANSFORMADORES={'SIH':transformar_sih,'SIM':transformar_sim,'CNES_ST':transformar_st,'CNES_LT':transformar_lt}


In [ ]:
from pyspark.storagelevel import StorageLevel
def existe(t):return spark.catalog.tableExists(t)
def ids(t):
 if not existe(t):return set()
 assert 'arquivo_id' in spark.table(t).columns, f'Tabela legada sem arquivo_id: {t}'
 return {str(x['arquivo_id']) for x in spark.table(t).select('arquivo_id').distinct().collect() if x['arquivo_id']}
def spath(p):
 p=str(p)
 return p[len('/lakehouse/default/'):] if p.startswith('/lakehouse/default/') else p
selecionados=df_manifesto[df_manifesto.fonte.isin(FONTES_EXECUCAO)].copy()
selecionados['prioridade']=selecionados.fonte.map({k:i for i,k in enumerate(ORDEM)})
selecionados=selecionados.sort_values(['prioridade','ano','uf','mes'])
processados={f:ids(TABELAS[f]) for f in ORDEM}
pendentes=[r for r in selecionados.to_dict('records') if r['arquivo_id'] not in processados[r['fonte']]]
if MAX_ARQUIVOS_EXECUCAO is not None:pendentes=pendentes[:MAX_ARQUIVOS_EXECUCAO]
print('Pendentes selecionados:',len(pendentes),'Gravacao:',APLICAR_GRAVACAO)
if APLICAR_GRAVACAO and not CONFIRMO_LAKEHOUSE_ORIGINAL:raise RuntimeError('Confirme lh_media_analytics antes de gravar')
resultado=[]
for fonte in ORDEM:
 ff=[x for x in pendentes if x['fonte']==fonte]
 for start in range(0,len(ff),ARQUIVOS_POR_LOTE):
  lote=ff[start:start+ARQUIVOS_POR_LOTE];frames=[];exp={}
  for it in lote:
   d=spark.read.parquet(spath(it['arquivo_bronze']))
   frames.append(TRANSFORMADORES[fonte](d,it));exp[it['arquivo_id']]=int(it['linhas_bronze'])
  df=frames[0]
  for d in frames[1:]:df=df.unionByName(d)
  df=df.persist(StorageLevel.DISK_ONLY)
  try:
   count={r['arquivo_id']:r['count'] for r in df.groupBy('arquivo_id').count().collect()}
   if count!=exp:raise RuntimeError(f'Contagem Bronze/Silver divergente: {count} != {exp}')
   tabela=TABELAS[fonte]
   if APLICAR_GRAVACAO:
    if set(exp)&ids(tabela):raise RuntimeError('Arquivo ja gravado; pare e reconcilie')
    if existe(tabela):
     schema_antigo={f.name:f.dataType.simpleString() for f in spark.table(tabela).schema.fields}
     schema_novo={f.name:f.dataType.simpleString() for f in df.schema.fields}
     if schema_antigo!=schema_novo:raise RuntimeError('Schema diferente; migracao necessaria')
     df.select(*spark.table(tabela).columns).write.format('delta').mode('append').saveAsTable(tabela)
    else:df.write.format('delta').mode('errorifexists').saveAsTable(tabela)
   resultado.append(dict(fonte=fonte,arquivos=len(lote),linhas=sum(exp.values()),status='CONCLUIDO' if APLICAR_GRAVACAO else 'SIMULADO'))
   print(resultado[-1])
  finally:df.unpersist()
display(pd.DataFrame(resultado))


In [ ]:
aud=[]
for fonte,tabela in TABELAS.items():
 manifest=df_manifesto[df_manifesto.fonte==fonte]
 contas={}
 if existe(tabela):contas={str(r['arquivo_id']):int(r['count']) for r in spark.table(tabela).groupBy('arquivo_id').count().collect()}
 for r in manifest.to_dict('records'):
  b=int(r['linhas_bronze']);s=contas.get(r['arquivo_id'],0)
  aud.append(dict(fonte=fonte,uf=r['uf'],ano=int(r['ano']),mes=int(r['mes']),arquivo_id=r['arquivo_id'],bronze=b,silver=s,status='OK' if b==s else ('PENDENTE' if s==0 else 'DIVERGENTE')))
audit=pd.DataFrame(aud)
display(audit.groupby(['fonte','status'],as_index=False).agg(arquivos=('arquivo_id','count'),bronze=('bronze','sum'),silver=('silver','sum')))
controle=ROOT/'controle'/'silver_v2';controle.mkdir(parents=True,exist_ok=True)
audit.to_parquet(controle/'reconciliacao.parquet',index=False)
print('PENDENTES:',sum(audit.status!='OK'))
